# env preparation

In [1]:
# adapt to colab gpu / nvidia 5060laptop setup

import os
import sys
import torch
import warnings

# SAVED_MODEL_NAME = "baseline_model_focal"
# LOADED_MODEL_NAME = "baseline_model_focal"
SAVED_MODEL_NAME = "baseline_injected_focal"
LOADED_MODEL_NAME = "lstm_model"
MODEL_NAME = "DeepPavlov/rubert-base-cased-sentence"
MAX_LEN = 512
DEBUG = False

try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    print("We are in Colab!")

    # Mount Drive
    from google.colab import drive
    drive.mount('/content/drive')

    !pip install -q transformers datasets seqeval evaluate accelerate torchcrf -q

    ROOT_DIR = "/content/drive/MyDrive/omg_diploma_2025/restore_punct"
    DATA_DIR = os.path.join(ROOT_DIR, "data")
    MODEL_DIR = os.path.join(ROOT_DIR, "models")
    # MODEL_DIR = DATA_DIR
    BATCH_SIZE = 8
    NUM_WORKERS = 2
    GRAD_ACCUM = 2

    if ROOT_DIR not in sys.path:
        sys.path.append(ROOT_DIR)

else:
    print("We are running locally!")
    ROOT_DIR = os.getcwd()
    DATA_DIR = os.path.join(ROOT_DIR, "data")
    MODEL_DIR = os.path.join(ROOT_DIR, "models")
    # MODEL_DIR = DATA_DIR
    # RTX 5060 (8GB VRAM)
    BATCH_SIZE = 4
    NUM_WORKERS = 4
    GRAD_ACCUM = 4

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)


We are running locally!


In [5]:
import json
import evaluate
import numpy as np
from transformers import AutoTokenizer, AutoConfig, AutoModel, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
from transformers.modeling_outputs import TokenClassifierOutput
from datasets import Dataset, load_dataset
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import KFold
from torchcrf import CRF
import torch.nn as nn
import pandas as pd


In [2]:
print(torch.cuda.is_available())
# !nvidia-smi


True


In [3]:
warnings.filterwarnings("ignore", category=FutureWarning, message=".*`tokenizer` is deprecated.*")
warnings.filterwarnings("ignore", message=".*seems not to be NE tag.*")
warnings.filterwarnings("ignore", message=".*UndefinedMetricWarning.*")


In [5]:
data_files = {
    "train": os.path.join(DATA_DIR, "train_all.json"),
    "validation": os.path.join(DATA_DIR, "val_internal.json")
}

raw_datasets = load_dataset("json", data_files=data_files)
print(raw_datasets)


DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 124488
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 6130
    })
})


In [6]:
with open(os.path.join(DATA_DIR, "label_map.json"), 'r', encoding='utf-8') as f:
    ID_TO_LABEL_RAW = json.load(f)
    id2label = {int(k): v for k, v in ID_TO_LABEL_RAW.items()}
    label2id = {v: k for k, v in id2label.items()}

LABELS = list(id2label.values())
NUM_LABELS = len(LABELS)

print(f"Loaded {NUM_LABELS} labels.")
# print(LABELS)

Loaded 28 labels.


In [8]:
def align_labels_with_tokens(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LEN
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # Special tokens
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx]) # First subword gets label
            else:
                label_ids.append(-100) # Subsequent subwords ignored
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


In [8]:
"""
looks like this:
{
  "tokens": ["Hello", ",", "world", "!"],
  "ner_tags": [0, 1, 0, 2]
}
"""

'\nlooks like this:\n{\n  "tokens": ["Hello", ",", "world", "!"],\n  "ner_tags": [0, 1, 0, 2]\n}\n'

# Model preparation

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
seqeval = evaluate.load("seqeval")

from custom_trainer import FocalLossTrainer, PUNCT_WEIGHTS, CRFTrainer

data_collator = DataCollatorForTokenClassification(tokenizer)

In [6]:
class BertCRFForTokenClassification(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.num_labels = num_labels
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        sequence_output = self.dropout(outputs.last_hidden_state)
        emissions = self.classifier(sequence_output)

        if labels is not None:
            mask = attention_mask.type(torch.uint8)
            safe_labels = torch.where(labels == -100, torch.zeros_like(labels), labels)
            loss = -self.crf(emissions, safe_labels, mask=mask, reduction='mean')
            return (loss, emissions)
        else:
            mask = attention_mask.type(torch.uint8) if attention_mask is not None else None
            predictions = self.crf.decode(emissions, mask=mask)
            return emissions

In [14]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        LABELS[p]
        for prediction, label in zip(predictions, labels)
        for p, l in zip(prediction, label)
        if l != -100
    ]

    true_labels = [
        LABELS[l]
        for prediction, label in zip(predictions, labels)
        for p, l in zip(prediction, label)
        if l != -100
    ]

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, true_predictions, average='weighted', zero_division=0
    )
    accuracy = accuracy_score(true_labels, true_predictions)

    report = classification_report(true_labels, true_predictions, zero_division=0)
    print("\n" + report)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy,
    }

# Training Pipeline

In [ ]:
#  TRAINING FLAGS # 
TRAIN_BASELINE = True
USE_CRF        = False
USE_LSTM       = False
USE_KFOLD      = False
TRAIN_FINETUNED = False
# # # # # # # # # # 

if DEBUG:
    NUM_EPOCHS    = 1
    FT_EPOCHS     = 1
    logging_steps = 5
else:
    NUM_EPOCHS    = 1
    FT_EPOCHS     = 1
    logging_steps = 50

suffix = "crf" if USE_CRF else "focal"
# BASELINE_MODEL_DIR  = os.path.join(MODEL_DIR, f"baseline_model_{suffix}")
# BASELINE_MODEL_DIR = os.path.join(MODEL_DIR, "baseline_model_crf")
# FINETUNED_MODEL_DIR = os.path.join(MODEL_DIR, f"finetuned_lstm_gera_{suffix}")
BASELINE_MODEL_DIR = os.path.join(MODEL_DIR, "finetuned_lstm_gera_lorugec_focal")
FINETUNED_MODEL_DIR = os.path.join(MODEL_DIR, "finetuned_lstm_gera_lorugec_focal")

print(f"TRAIN_BASELINE={TRAIN_BASELINE}  USE_CRF={USE_CRF}  USE_KFOLD={USE_KFOLD}")
print(f"Baseline  → {BASELINE_MODEL_DIR}")
print(f"Finetuned → {FINETUNED_MODEL_DIR}")

TRAIN_BASELINE=True  USE_CRF=False  USE_KFOLD=False
Baseline  → /home/temari/god please no diploma/restore_punct/models/finetuned_injected2_focal
Finetuned → /home/temari/god please no diploma/restore_punct/models/finetuned_injected2_focal


In [13]:
class BertLSTMForTokenClassification(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.num_labels = num_labels
        self.bert = AutoModel.from_pretrained(model_name)
        
        self.lstm = nn.LSTM(
            input_size=self.bert.config.hidden_size, 
            hidden_size=256, 
            batch_first=True, 
            bidirectional=True
        )
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(256 * 2, num_labels)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        sequence_output = outputs.last_hidden_state
        
        lstm_output, _ = self.lstm(sequence_output)
        lstm_output = self.dropout(lstm_output)
        
        logits = self.classifier(lstm_output)

        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            return (loss, logits)
        else:
            return logits


In [16]:
def get_model(use_crf=USE_CRF, use_lstm=USE_LSTM, load_from_path=None):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    if use_lstm:
        model = BertLSTMForTokenClassification(MODEL_NAME, num_labels=NUM_LABELS)
    elif use_crf:
        model = BertCRFForTokenClassification(MODEL_NAME, num_labels=NUM_LABELS)
    else:
        model = AutoModelForTokenClassification.from_pretrained(
            MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id
        )

    if load_from_path is not None:
        if use_crf or use_lstm:
            file_name = "lstm_weights.pth" if use_lstm else "pytorch_model.bin"
            weights_path = os.path.join(load_from_path, file_name)
            model.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
        else:
            model = AutoModelForTokenClassification.from_pretrained(
                load_from_path, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id
            )
            
    return model.to(device)


In [ ]:
def get_trainer(model_to_train, train_ds, val_ds,
                output_dir="training-output", learning_rate=2e-5,
                num_epochs=3, eval_strategy="epoch", save_strategy="epoch"):
    
    args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy=eval_strategy,
        save_strategy=save_strategy,
        learning_rate=learning_rate,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        logging_steps=logging_steps,
        gradient_accumulation_steps=GRAD_ACCUM,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    if USE_CRF:
        from custom_trainer import CRFTrainer
        return CRFTrainer(
            model=model_to_train, args=args, train_dataset=train_ds,
            eval_dataset=val_ds, processing_class=tokenizer,
            data_collator=data_collator, compute_metrics=compute_metrics,
        )
        
    elif USE_LSTM:
        from transformers import Trainer
        return Trainer(
            model=model_to_train, args=args, train_dataset=train_ds,
            eval_dataset=val_ds, processing_class=tokenizer,
            data_collator=data_collator, compute_metrics=compute_metrics,
        )
        
    else:
        # === ВОТ ЗДЕСЬ ДОБАВЛЯЕТСЯ АЛЬФА ===
        # Создаем тензор весов длиной NUM_LABELS. По умолчанию ставим 1.0.
        alpha_tensor = torch.ones(NUM_LABELS)
        
        # Заполняем его значениями из твоего словаря PUNCT_WEIGHTS
        for label_name, weight in PUNCT_WEIGHTS.items():
            if label_name in label2id:
                label_idx = label2id[label_name]
                alpha_tensor[label_idx] = weight

        return FocalLossTrainer(
            model=model_to_train,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            processing_class=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
            alpha_tensor=alpha_tensor # Передаем тензор в тренер!
        )

In [12]:
def save_model(model_to_save, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    if USE_CRF or USE_LSTM:
        torch.save(
            model_to_save.state_dict(),
            os.path.join(save_dir, "pytorch_model.bin"),
        )
    else:
        model_to_save.save_pretrained(save_dir)
        
    tokenizer.save_pretrained(save_dir)
    print(f"Model saved → {save_dir}")


## Baseline Training

In [17]:
tokenized_baseline = raw_datasets.map(align_labels_with_tokens, batched=True)

if DEBUG:
    tokenized_baseline["train"] = tokenized_baseline["train"].select(range(50))
    tokenized_baseline["validation"] = tokenized_baseline["validation"].select(range(10))

if TRAIN_BASELINE:
    print(f"Training baseline ({'CRF' if USE_CRF else 'Focal Loss'})…")

    model_baseline = get_model(use_crf=USE_CRF)

    trainer_baseline = get_trainer(
        model_baseline,
        tokenized_baseline["train"],
        tokenized_baseline["validation"],
        output_dir=os.path.join(MODEL_DIR, f"bert-{suffix}-baseline_temp"),
        learning_rate=2e-5,
        num_epochs=NUM_EPOCHS,
        save_strategy="no"
    )
    trainer_baseline.train()

    save_model(model_baseline, BASELINE_MODEL_DIR)

    del model_baseline, trainer_baseline
    torch.cuda.empty_cache()
else:
    print(f"Skipping baseline (TRAIN_BASELINE=False).\n"
          f"Expecting saved model at: {BASELINE_MODEL_DIR}")

Map:   0%|          | 0/124488 [00:00<?, ? examples/s]

Map:   0%|          | 0/6130 [00:00<?, ? examples/s]

Training baseline (Focal Loss)…


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: DeepPavlov/rubert-base-cased-sentence
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.134268,0.027378,0.959068,0.960727,0.959479,0.960727



              precision    recall  f1-score   support

           "       0.88      0.72      0.79     12048
           !       0.32      0.04      0.07       193
         ! -       0.00      0.00      0.00        15
          "        0.83      0.75      0.79      8086
          ""       0.74      0.29      0.42        69
          ",       0.77      0.62      0.69      2353
        ", -       0.72      0.88      0.79      1559
          ".       0.76      0.62      0.68      3542
          "?       0.00      0.00      0.00         7
           ,       0.88      0.89      0.89     66077
         , "       0.65      0.47      0.55       578
         , -       0.73      0.51      0.60       823
           -       0.68      0.52      0.59      7014
         - "       0.35      0.27      0.30       237
           .       0.92      0.96      0.94     49971
         . "       0.69      0.71      0.70      2247
         . -       0.75      0.59      0.66       972
           :       0.69   

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved → /home/temari/god please no diploma/restore_punct/models/baseline_injected_focal


In [29]:
# tmp
print("Continuing baseline training for 2 more epochs from 1-epoch checkpoint…")

model_continue = get_model(use_crf=USE_CRF, load_from_path=BASELINE_MODEL_DIR)

trainer_continue = get_trainer(
    model_continue,
    tokenized_baseline["train"],
    tokenized_baseline["validation"],
    output_dir=os.path.join(MODEL_DIR, "bert-focal-baseline-continue_temp"),
    learning_rate=2e-5,
    num_epochs=2,
    save_strategy="no"
)
trainer_continue.train()

save_model(model_continue, BASELINE_MODEL_DIR)

del model_continue, trainer_continue
torch.cuda.empty_cache()
print("Done — 3 total epochs completed.")

Continuing baseline training for 2 more epochs from 1-epoch checkpoint…


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: DeepPavlov/rubert-base-cased-sentence
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.118849,0.024558,0.962094,0.963717,0.962439,0.963717
2,0.098760,0.023658,0.963596,0.964901,0.963958,0.964901



              precision    recall  f1-score   support

           "       0.89      0.73      0.80     12048
           !       0.51      0.11      0.19       193
         ! -       0.00      0.00      0.00        15
          "        0.86      0.75      0.80      8086
          ""       0.59      0.35      0.44        69
          ",       0.77      0.65      0.71      2353
        ", -       0.78      0.87      0.83      1559
          ".       0.79      0.65      0.71      3542
          "?       0.00      0.00      0.00         7
           ,       0.89      0.90      0.89     66077
         , "       0.69      0.48      0.57       578
         , -       0.72      0.63      0.67       823
           -       0.72      0.52      0.60      7014
         - "       0.39      0.30      0.34       237
           .       0.93      0.96      0.95     49971
         . "       0.73      0.72      0.72      2247
         . -       0.72      0.72      0.72       972
           :       0.71   

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved → /home/temari/god please no diploma/restore_punct/models/baseline_injected_focal
Done — 3 total epochs completed.


## LORuGEC / GERA Fine-Tuning

In [18]:
# lorugec_files = {
#     "lorugec_train_all":  os.path.join(DATA_DIR, "lorugec_train_all.json"),
#     "train":      os.path.join(DATA_DIR, "train_lorugec.json"),
#     "validation": os.path.join(DATA_DIR, "val_lorugec.json"),
#     "test":       os.path.join(DATA_DIR, "bench_test_lorugec.json"),
# }

# raw_lorugec = load_dataset("json", data_files=lorugec_files)
# tokenized_lorugec = raw_lorugec.map(align_labels_with_tokens, batched=True)
# print(raw_lorugec)

if not TRAIN_FINETUNED:
    tokenized_lorugec = None
    print("Skipping fine-tuning (TRAIN_FINETUNED=False).")
else:
    gera_files = {
        "train":      os.path.join(DATA_DIR, "train_gera.json"),
        "validation": os.path.join(DATA_DIR, "val_gera.json"),
        "test":       os.path.join(DATA_DIR, "bench_test_gera.json"),
    }

    raw_lorugec = load_dataset("json", data_files=gera_files)
    tokenized_lorugec = raw_lorugec.map(align_labels_with_tokens, batched=True)
    print(raw_lorugec)


Skipping fine-tuning (TRAIN_FINETUNED=False).


In [19]:
# # option with mixing some previous data into lorugec train set

# from datasets import concatenate_datasets

# lorugec_files = {
#     "lorugec_train_all":  os.path.join(DATA_DIR, "lorugec_train_all.json"),
#     "train":      os.path.join(DATA_DIR, "train_lorugec.json"),
#     "validation": os.path.join(DATA_DIR, "val_lorugec.json"),
#     "test":       os.path.join(DATA_DIR, "bench_test_lorugec.json"),
# }
# raw_lorugec = load_dataset("json", data_files=lorugec_files)

# baseline_dataset = load_dataset("json", data_files={"train": os.path.join(DATA_DIR, "train_all.json")})
# baseline_sample = baseline_dataset["train"].shuffle(seed=42).select(range(3000))

# mixed_train_all = concatenate_datasets([raw_lorugec["lorugec_train_all"], baseline_sample])
# mixed_train_simple = concatenate_datasets([raw_lorugec["train"], baseline_sample])

# raw_lorugec["lorugec_train_all"] = mixed_train_all.shuffle(seed=42)
# raw_lorugec["train"] = mixed_train_simple.shuffle(seed=42)

# print("Dataset Sizes after mixing in Baseline Memory:")
# print(raw_lorugec)

# tokenized_lorugec = raw_lorugec.map(align_labels_with_tokens, batched=True)

In [20]:
if not TRAIN_FINETUNED:
    print("Skipping fine-tuning training (TRAIN_FINETUNED=False).")
else:
    FT_LR   = 1e-5
    N_FOLDS = 5
    
    if USE_KFOLD:
        kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
        full_ds = tokenized_lorugec["train"]
        fold_results = []
    
        for fold, (train_idx, val_idx) in enumerate(kf.split(range(len(full_ds)))):
            print(f"\n{'='*50}")
            print(f"  Fold {fold + 1}/{N_FOLDS}")
            print(f"{'='*50}")
    
            model_fold = get_model(use_crf=USE_CRF, load_from_path=BASELINE_MODEL_DIR)
    
            trainer_fold = get_trainer(
                model_fold,
                full_ds.select(train_idx),
                full_ds.select(val_idx),
                os.path.join(MODEL_DIR, f"bert-lorugec-fold{fold}_temp"),
                learning_rate=FT_LR,
                num_epochs=FT_EPOCHS,
                save_strategy="no",
            )
            trainer_fold.train()
    
            metrics = trainer_fold.evaluate()
            print(f"  → F1={metrics['eval_f1']:.4f}  "
                  f"P={metrics['eval_precision']:.4f}  "
                  f"R={metrics['eval_recall']:.4f}")
            fold_results.append(metrics)
    
            del model_fold, trainer_fold
            torch.cuda.empty_cache()
    
        print(f"\n{'='*50}")
        print("  KFold Summary")
        print(f"{'='*50}")
        for key in ("eval_f1", "eval_precision", "eval_recall", "eval_accuracy"):
            vals = [r[key] for r in fold_results]
            print(f"  {key:20s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")
        print(f"{'='*50}")
    
        # Final model on lorugec_train_all
        print("\nTraining final production model on 100% of lorugec_train_all…")
        model_final = get_model(use_crf=USE_CRF, load_from_path=BASELINE_MODEL_DIR)
    
        trainer_final = get_trainer(
            model_final, full_ds, full_ds,
            output_dir=os.path.join(MODEL_DIR, "bert-lorugec-final_temp"),
            learning_rate=FT_LR,
            num_epochs=FT_EPOCHS,
            eval_strategy="no",
            save_strategy="no",
        )
        trainer_final.train()
        save_model(model_final, FINETUNED_MODEL_DIR)
    
        del model_final, trainer_final
        torch.cuda.empty_cache()
    
    else:
        # Simple train / val split
        print("Simple fine-tuning (train / val split)…")
        model_ft = get_model(use_crf=USE_CRF, load_from_path=BASELINE_MODEL_DIR)
    
        trainer_ft = get_trainer(
            model_ft,
            tokenized_lorugec["train"],
            tokenized_lorugec["validation"],
            output_dir=os.path.join(MODEL_DIR, "bert-lorugec-simple_temp"),
            learning_rate=FT_LR,
            num_epochs=FT_EPOCHS,
        )
        trainer_ft.train()
        save_model(model_ft, FINETUNED_MODEL_DIR)
    
        del model_ft, trainer_ft
        torch.cuda.empty_cache()

Skipping fine-tuning training (TRAIN_FINETUNED=False).


## LORuGEC Benchmark

In [21]:
if not TRAIN_FINETUNED:
    print("Skipping benchmark (TRAIN_FINETUNED=False).")
else:
    print(f"Loading final model from {FINETUNED_MODEL_DIR}…")
    model_bench = get_model(use_crf=USE_CRF, load_from_path=FINETUNED_MODEL_DIR)

    trainer_bench = get_trainer(
        model_bench,
        tokenized_lorugec["test"],
        tokenized_lorugec["test"],
        output_dir=os.path.join(MODEL_DIR, "benchmark-eval_temp"),
        eval_strategy="no",
        save_strategy="no",
    )

    bench = trainer_bench.evaluate(tokenized_lorugec["test"])

    print(f"\n{'='*50}")
    print("  OFFICIAL BENCHMARK  (bench_test_lorugec.json)")
    print(f"{'='*50}")
    print(f"  Precision : {bench['eval_precision']:.4f}")
    print(f"  Recall    : {bench['eval_recall']:.4f}")
    print(f"  F1        : {bench['eval_f1']:.4f}")
    print(f"  Accuracy  : {bench['eval_accuracy']:.4f}")
    print(f"{'='*50}")

Skipping benchmark (TRAIN_FINETUNED=False).


# Restoration Demo

In [17]:
import re

model_dir = FINETUNED_MODEL_DIR if TRAIN_FINETUNED else BASELINE_MODEL_DIR
model = get_model(use_crf=USE_CRF, load_from_path=model_dir)
model.eval()


def reconstruct_text(tokens, labels, id2label):
    text = ""
    for token, label_id in zip(tokens, labels):
        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue
        punct = id2label[label_id]
        if punct == "O":
            text += token + " "
        else:
            text += token + punct + " "
    return text.strip()


def restore_punctuation(text):
    device = next(model.parameters()).device

    words = [m.group() for m in re.finditer(r"[\w]+(?:-[\w]+)*", text)]
    if not words:
        return ""

    inputs = tokenizer(
        words,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
        is_split_into_words=True,
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

        if hasattr(model, "crf"):
            emissions = outputs[1] if isinstance(outputs, tuple) else outputs
            mask = inputs["attention_mask"].byte()
            predictions = model.crf.decode(emissions, mask=mask)[0]
        else:
            logits = outputs.logits if hasattr(outputs, "logits") else outputs
            predictions = torch.argmax(logits, dim=2)[0].cpu().tolist()

    word_ids = inputs.word_ids()
    restored_text = ""
    previous_word_idx = None

    for i, word_idx in enumerate(word_ids):
        if word_idx is None or word_idx == previous_word_idx:
            continue
        original_word = words[word_idx]
        pred_id = predictions[i] if i < len(predictions) else 0
        punct = id2label.get(pred_id, "O")

        if punct == "O":
            restored_text += original_word + " "
        else:
            restored_text += original_word + punct + " "
        previous_word_idx = word_idx

    return re.sub(r'\s+', ' ', restored_text).strip()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: DeepPavlov/rubert-base-cased-sentence
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [18]:
tests = [
    "Мама мыла раму а папа читал газету",
    "Однако мы решили не идти в кино потому что пошел дождь",
    "Он сказал Привет как дела",
    "Я говорю ей Что думаешь А она мне Да ничего отстань уже",
    "После многих страданий А А Пушкин всё-таки написал свои стишки",
]

for t in tests:
    print(f"Input:    {t}")
    print(f"Restored: {restore_punctuation(t)}\n")

Input:    Мама мыла раму а папа читал газету
Restored: Мама мыла раму, а папа читал газету.

Input:    Однако мы решили не идти в кино потому что пошел дождь
Restored: Однако мы решили не идти в кино, потому что пошел дождь.

Input:    Он сказал Привет как дела
Restored: Он сказал: " Привет, как дела.

Input:    Я говорю ей Что думаешь А она мне Да ничего отстань уже
Restored: Я говорю ей: " Что думаешь А она мне Да ничего отстань уже.

Input:    После многих страданий А А Пушкин всё-таки написал свои стишки
Restored: После многих страданий А. А. Пушкин всё-таки написал свои стишки.



# Saving test results

In [24]:
import os
import json
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
from tqdm.auto import tqdm
import datetime


In [25]:
MODELS_TO_EVALUATE =[
    {
        "name": "baseline_injected_focal", 
        "path": os.path.join(MODEL_DIR, "baseline_injected_focal"), 
        "use_crf": False
    # },
    # {
    #     "name": "baseline_model_crf", 
    #     "path": os.path.join(MODEL_DIR, "baseline_model_crf"), 
    #     "use_crf": True
    # },
    # {
    #     "name": "bert-lorugec-final", 
    #     "path": os.path.join(MODEL_DIR, "bert-lorugec-final"), 
    #     "use_crf": True
    # },
    # {
    #     "name": "baseline_model", 
    #     "path": os.path.join(MODEL_DIR, "baseline_model"), 
    #     "use_crf": False
    # },
    # {
    #     "name": "finetuned_lorugec_crf", 
    #     "path": os.path.join(MODEL_DIR, "finetuned_lorugec_crf"), 
    #     "use_crf": True
    }
]

TEST_DATASETS = {
    "General_Test": os.path.join(DATA_DIR, "bench_test_all.json"),
    "LORuGEC_Test": os.path.join(DATA_DIR, "bench_test_lorugec.json")
}
# # # # # # # # #
# path to save! #
RESULTS_DB_PATH = os.path.join(ROOT_DIR, "results_injected_focal.json")
EXCEL_OUTPUT_PATH = os.path.join(ROOT_DIR, "results_injected_focal.xlsx")


In [26]:
# inference loop
def evaluate_model_on_data(model, dataset, use_crf=False):
    model.eval()
    device = next(model.parameters()).device
    
    # FIX: Remove string columns so the collator doesn't crash trying to turn text into tensors
    valid_cols =["input_ids", "attention_mask", "token_type_ids", "labels"]
    cols_to_remove =[col for col in dataset.column_names if col not in valid_cols]
    eval_dataset = dataset.remove_columns(cols_to_remove)
    
    all_preds, all_labels = [],[]
    dataloader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, collate_fn=data_collator)
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")
            
            if use_crf:
                emissions = model(**batch)
                mask = batch["attention_mask"].byte()
                # CRF decode returns a list of lists containing the predicted sequences
                preds = model.crf.decode(emissions, mask=mask)
                
                for i in range(len(preds)):
                    for j, p in enumerate(preds[i]):
                        l = labels[i][j].item()
                        if l != -100: # Ignore special tokens and subwords
                            all_preds.append(p)
                            all_labels.append(l)
            else:
                outputs = model(**batch)
                logits = outputs.logits if hasattr(outputs, "logits") else outputs[1] if isinstance(outputs, tuple) else outputs
                preds = torch.argmax(logits, dim=2)
                
                for i in range(labels.size(0)):
                    for j in range(labels.size(1)):
                        l = labels[i, j].item()
                        if l != -100: # Ignore special tokens and subwords
                            all_preds.append(preds[i, j].item())
                            all_labels.append(l)
                            
    true_tags = [id2label[l] for l in all_labels]
    pred_tags = [id2label[p] for p in all_preds]
    
    return classification_report(true_tags, pred_tags, output_dict=True, zero_division=0)

In [27]:
# evaluation and saving to json
if os.path.exists(RESULTS_DB_PATH):
    with open(RESULTS_DB_PATH, "r", encoding="utf-8") as f:
        results_db = json.load(f)
else:
    results_db = {}

for model_info in MODELS_TO_EVALUATE:
    name = model_info["name"]
    path = model_info["path"]
    use_crf = model_info["use_crf"]
    
    if not os.path.exists(path):
        print(f"Skipping {name}: Path does not exist ({path})")
        continue
        
    print(f"\n[{name}] Loading model...")
    model = get_model(use_crf=use_crf, load_from_path=path)
    
    if name not in results_db:
        results_db[name] = {"timestamp": str(datetime.datetime.now())}
        
    for test_name, test_file in TEST_DATASETS.items():
        print(f"[{name}] Evaluating on {test_name}...")
        raw_test = load_dataset("json", data_files={"test": test_file})
        tok_test = raw_test.map(align_labels_with_tokens, batched=True)["test"]
        
        report = evaluate_model_on_data(model, tok_test, use_crf)
        results_db[name][test_name] = report
        
    del model
    torch.cuda.empty_cache()

with open(RESULTS_DB_PATH, "w", encoding="utf-8") as f:
    json.dump(results_db, f, indent=4, ensure_ascii=False)
print(f"\nSaved raw results to JSON: {RESULTS_DB_PATH}")



[baseline_injected_focal] Loading model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: DeepPavlov/rubert-base-cased-sentence
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[baseline_injected_focal] Evaluating on General_Test...


...

[baseline_injected_focal] Evaluating on LORuGEC_Test...


Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/603 [00:00<?, ? examples/s]

Evaluating:   0%|          | 0/151 [00:00<?, ?it/s]


Saved raw results to JSON: /home/temari/god please no diploma/restore_punct/results_injected_focal.json


In [28]:
def export_thesis_tables(db):
    summary_records = []
    detailed_records =[]
    
    for model_name, data in db.items():
        sum_row = {"Model": model_name}
        for test_name in TEST_DATASETS.keys():
            if test_name in data and "weighted avg" in data[test_name]:
                metrics = data[test_name]["weighted avg"]
                sum_row[(test_name, "F1")] = round(metrics["f1-score"] * 100, 2)
                sum_row[(test_name, "Precision")] = round(metrics["precision"] * 100, 2)
                sum_row[(test_name, "Recall")] = round(metrics["recall"] * 100, 2)
        summary_records.append(sum_row)
        
        labels = set()
        for test_name in TEST_DATASETS.keys():
            if test_name in data:
                labels.update([k for k in data[test_name].keys() if k not in["accuracy", "macro avg", "weighted avg", "timestamp"]])
        
        for label in sorted(labels):
            det_row = {"Model": model_name, "Punctuation": label}
            for test_name in TEST_DATASETS.keys():
                if test_name in data and label in data[test_name]:
                    metrics = data[test_name][label]
                    det_row[(test_name, "F1")] = round(metrics["f1-score"] * 100, 2)
                    det_row[(test_name, "Precision")] = round(metrics["precision"] * 100, 2)
                    det_row[(test_name, "Recall")] = round(metrics["recall"] * 100, 2)
                    det_row[(test_name, "Support")] = metrics["support"]
            detailed_records.append(det_row)

    df_summary = pd.DataFrame(summary_records)
    df_detailed = pd.DataFrame(detailed_records)

    if not df_summary.empty:
        df_summary.set_index("Model", inplace=True)
        df_summary.columns = pd.MultiIndex.from_tuples(df_summary.columns)

    if not df_detailed.empty:
        df_detailed.set_index(["Model", "Punctuation"], inplace=True)
        df_detailed.columns = pd.MultiIndex.from_tuples(df_detailed.columns)

    with pd.ExcelWriter(EXCEL_OUTPUT_PATH, engine='openpyxl') as writer:
        if not df_summary.empty:
            df_summary.to_excel(writer, sheet_name="Summary")
        if not df_detailed.empty:
            df_detailed.to_excel(writer, sheet_name="Per-Class Details")

    print(f"Saved formatted Excel workbook to: {EXCEL_OUTPUT_PATH}")
    print(" -> Check the tabs at the bottom of the Excel file!")

export_thesis_tables(results_db)


Saved formatted Excel workbook to: /home/temari/god please no diploma/restore_punct/results_injected_focal.xlsx
 -> Check the tabs at the bottom of the Excel file!
